<a href="https://colab.research.google.com/github/MuhammaadArsalanZahid/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

**FlyRank Machine Learning — Week 5 Build**

This notebook trains a transparent ML model and compares it with the Week-4 baseline on the same held-out data.

Required: method choice, valid split, train + baseline comparison, useful metrics, feature interpretation, error analysis, and self-checks.


In [2]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd()
candidates = [
    ROOT / "data/raw/content_refresh_anonymized.csv",
    ROOT / "content_refresh_anonymized.csv",
    Path("/content/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Upload content_refresh_anonymized.csv to Colab or place it in data/raw/."
    )

df = pd.read_csv(data_path)
print("Dataset:", data_path)
print("Shape:", df.shape)


Dataset: /content/content_refresh_anonymized.csv
Shape: (30000, 53)


## 1. Method choice and why

**Random Forest** is used because it can capture non-linear relationships among current-snapshot signals while remaining practical to train and interpret. The model must beat the Week-4 baseline on the same test set; complexity alone is not a success criterion.


In [3]:

def find_col(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    return None

ctr_col = find_col(df, ["ctr", "CTR", "avg_ctr", "average_ctr"])
position_col = find_col(df, ["avg_position", "position", "average_position"])
impressions_col = find_col(
    df, ["impressions_90d", "impressions", "avg_impressions", "impressions_90_days"]
)
client_col = find_col(df, ["client_hash_id", "client_id", "client"])

required = {
    "CTR": ctr_col,
    "position": position_col,
    "impressions": impressions_col,
    "client/group": client_col
}
missing = [k for k,v in required.items() if v is None]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

for c in [ctr_col, position_col, impressions_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Selected columns:")
for k,v in required.items():
    print(f"{k}: {v}")


Selected columns:
CTR: ctr
position: avg_position
impressions: impressions_90d
client/group: client_id


## 2. Target and feature contract

The model uses current descriptive signals only. Existing action/decision flags, identifiers, labels, and future-window fields are excluded from features.

If the dataset has no explicit binary outcome, the notebook uses a transparent current-snapshot proxy target aligned with the Week-4 prioritization idea. This is clearly reported as a proxy, not a future business outcome.


In [4]:

feature_cols = [ctr_col, position_col, impressions_col]

blocked = [
    "label","target","future","flag","needs_","is_","action",
    "quick_win","underperform","refresh","decision","id","hash"
]

extra = []
for c in df.select_dtypes(include=np.number).columns:
    lc = c.lower()
    if c in feature_cols:
        continue
    if any(t in lc for t in blocked):
        continue
    extra.append(c)

feature_cols += extra[:7]

label_candidates = ["label","target","y","outcome"]
label_col = find_col(df, label_candidates)

if label_col is not None:
    y0 = pd.to_numeric(df[label_col], errors="coerce")
    vals = set(y0.dropna().unique())
    if vals.issubset({0,1}) and len(vals) == 2:
        df["_target"] = y0.astype("Int64")
        target_description = f"Provided binary outcome: {label_col}"
    else:
        label_col = None

if label_col is None:
    ctr_median = df.loc[df[ctr_col] > 0, ctr_col].median()
    imp_q75 = df[impressions_col].quantile(.75)
    df["_target"] = (
        (df[impressions_col] >= imp_q75) &
        (df[ctr_col] > 0) &
        (df[ctr_col] <= ctr_median)
    ).astype(int)
    target_description = (
        "Proxy target: high visibility + below-median positive CTR "
        "(current snapshot only)"
    )

model_df = df.dropna(subset=feature_cols + [client_col, "_target"]).copy()
model_df["_target"] = model_df["_target"].astype(int)

print("Features:", feature_cols)
print("Target:", target_description)
print(model_df["_target"].value_counts().sort_index())


Features: ['ctr', 'avg_position', 'impressions_90d', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'clicks_90d', 'pageviews_90d']
Target: Proxy target: high visibility + below-median positive CTR (current snapshot only)
_target
0    16983
1     3035
Name: count, dtype: int64


## 3. Split design

A client-grouped holdout is used so a client cannot appear in both training and testing. This gives a cleaner test of generalization to unseen clients.


In [5]:

groups = model_df[client_col].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(model_df[feature_cols], model_df["_target"], groups=groups)
)

train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

train_clients = set(train[client_col].astype(str))
test_clients = set(test[client_col].astype(str))

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

assert len(train_clients & test_clients) == 0
print("PASS: zero client overlap.")


Train rows: 14242
Test rows: 5776
Train clients: 23
Test clients: 6
Client overlap: 0
PASS: zero client overlap.


## 4. Train Random Forest

In [6]:

X_train, y_train = train[feature_cols], train["_target"]
X_test, y_test = test[feature_cols], test["_target"]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
model.fit(X_train, y_train)

test = test.copy()
test["model_score"] = model.predict_proba(X_test)[:,1]
print("Random Forest trained.")


Random Forest trained.


## 5. Recreate Week-4 baseline on the same held-out data

Baseline thresholds are learned from training rows only and then applied to the same test rows used by the model.


In [7]:

train_ctr_median = train.loc[train[ctr_col] > 0, ctr_col].median()
train_imp_q75 = train[impressions_col].quantile(.75)

train_bucket = pd.cut(
    train[position_col],
    bins=[-np.inf,3,10,20,50,np.inf],
    labels=["1-3","4-10","11-20","21-50","51+"],
    include_lowest=True
)
position_medians = (
    train.assign(_bucket=train_bucket)
         .groupby("_bucket", observed=False)[ctr_col]
         .median()
)

test["_bucket"] = pd.cut(
    test[position_col],
    bins=[-np.inf,3,10,20,50,np.inf],
    labels=["1-3","4-10","11-20","21-50","51+"],
    include_lowest=True
)
test["_expected_ctr"] = test["_bucket"].map(position_medians).astype(float)
test["_ctr_gap"] = (test["_expected_ctr"] - test[ctr_col]).clip(lower=0).fillna(0)

test["baseline_score"] = np.where(
    (test[impressions_col] >= train_imp_q75) &
    (test[ctr_col] > 0) &
    (test[ctr_col] <= train_ctr_median),
    test[impressions_col] * test["_ctr_gap"],
    0.0
)

print("Baseline reconstructed from training-only statistics.")


Baseline reconstructed from training-only statistics.


## 6. Model vs baseline

Primary metric: **Precision@50** — the proportion of positive rows among the 50 highest-ranked rows. Both methods are evaluated on exactly the same held-out test rows.


In [8]:

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean()) if k else np.nan

def metrics(y_true, scores):
    out = {"Precision@50": precision_at_k(y_true, scores)}
    if len(np.unique(y_true)) == 2:
        out["ROC-AUC"] = roc_auc_score(y_true, scores)
        out["Average Precision"] = average_precision_score(y_true, scores)
    else:
        out["ROC-AUC"] = np.nan
        out["Average Precision"] = np.nan
    return out

baseline_metrics = metrics(test["_target"], test["baseline_score"])
model_metrics = metrics(test["_target"], test["model_score"])

comparison = pd.DataFrame([
    {"Method":"Week-4 Baseline", **baseline_metrics},
    {"Method":"Random Forest", **model_metrics},
])

display(comparison)

delta = model_metrics["Precision@50"] - baseline_metrics["Precision@50"]
print(f"Precision@50 delta (model - baseline): {delta:.4f}")


,Method,Precision@50,ROC-AUC,Average Precision
0,Week-4 Baseline,1.0,0.680505,0.452943
1,Random Forest,1.0,1.000000,1.000000


Precision@50 delta (model - baseline): 0.0000


## 7. Feature interpretation — permutation importance

In [9]:

if len(np.unique(y_test)) == 2:
    perm = permutation_importance(
        model, X_test, y_test,
        scoring="average_precision",
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )
    importance = (
        pd.DataFrame({
            "feature": feature_cols,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std
        })
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )
    display(importance)
else:
    importance = pd.DataFrame()
    print("Permutation importance skipped: test target contains one class.")


,feature,importance_mean,importance_std
0,impressions_90d,5.211939e-01,1.272294e-02
1,ctr,3.627324e-01,1.326542e-02
2,clicks_90d,1.159127e-05,1.105708e-05
3,avg_position,3.330669e-17,5.087681e-17
4,word_count,2.220446e-17,4.440892e-17
5,search_volume,2.220446e-17,4.440892e-17
6,char_count,1.110223e-17,3.330669e-17
7,cpc,1.110223e-17,3.330669e-17
8,competition,0.000000e+00,0.000000e+00
9,pageviews_90d,0.000000e+00,0.000000e+00


## 8. Errors and interpretation

In [10]:

test["predicted_class"] = (test["model_score"] >= .5).astype(int)
test["error_type"] = np.select(
    [
        (test["_target"] == 0) & (test["predicted_class"] == 1),
        (test["_target"] == 1) & (test["predicted_class"] == 0)
    ],
    ["False Positive","False Negative"],
    default="Correct"
)

error_cols = [
    c for c in [
        client_col,
        ctr_col,
        position_col,
        impressions_col,
        "model_score",
        "baseline_score",
        "_target",
        "error_type"
    ] if c in test.columns
]

print("Top false positives:")
display(
    test[test["error_type"]=="False Positive"]
    .sort_values("model_score", ascending=False)
    .head(10)[error_cols]
)

print("Top false negatives:")
display(
    test[test["error_type"]=="False Negative"]
    .sort_values("model_score", ascending=False)
    .head(10)[error_cols]
)

print(
    "\nInterpretation: high model scores do not prove an SEO action is correct. "
    "Query intent, SERP features, brand effects, seasonality and tracking noise "
    "can make individual predictions wrong. If the target is a proxy, the model "
    "is learning the proxy rather than proving future business impact."
)


Top false positives:


,client_id,ctr,avg_position,impressions_90d,model_score,baseline_score,_target,error_type


Top false negatives:


,client_id,ctr,avg_position,impressions_90d,model_score,baseline_score,_target,error_type



Interpretation: high model scores do not prove an SEO action is correct. Query intent, SERP features, brand effects, seasonality and tracking noise can make individual predictions wrong. If the target is a proxy, the model is learning the proxy rather than proving future business impact.


## 9. Self-check

In [11]:

feature_lower = [c.lower() for c in feature_cols]
forbidden_tokens = [
    "future","label","target","needs_","quick_win","underperform",
    "refresh","decision","action","flag","hash","client_id","content_id"
]
violations = [
    c for c in feature_lower
    if any(t in c for t in forbidden_tokens)
]

print("Features used:", feature_cols)
print("Potential forbidden feature names:", violations)

assert not violations
assert len(train_clients & test_clients) == 0
assert len(test) == len(y_test)
assert {"Method","Precision@50","ROC-AUC","Average Precision"}.issubset(comparison.columns)

print("PASS: no obvious forbidden feature names.")
print("PASS: grouped validation has zero client overlap.")
print("PASS: baseline and model use the same held-out rows.")
print("PASS: model-vs-baseline table exists.")
print("PASS: ML-08 self-check complete.")


Features used: ['ctr', 'avg_position', 'impressions_90d', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'clicks_90d', 'pageviews_90d']
Potential forbidden feature names: []
PASS: no obvious forbidden feature names.
PASS: grouped validation has zero client overlap.
PASS: baseline and model use the same held-out rows.
PASS: model-vs-baseline table exists.
PASS: ML-08 self-check complete.
